In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models

### Loading Dataset and Transform

In [4]:
train_path = "/Users/haroon/Developer/Fruit_Fresh_Rotten_ESP/dataset_Fruits/train"

test_path = "/Users/haroon/Developer/Fruit_Fresh_Rotten_ESP/dataset_Fruits/test"

train_ds = tf.keras.utils.image_dataset_from_directory(
    directory=train_path,
    image_size = (96, 96),
    color_mode='grayscale',
    batch_size=32,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    directory=test_path,
    image_size = (96,96),
    color_mode='grayscale',
    batch_size=32,
    shuffle=False
)


Found 4740 files belonging to 3 classes.
Found 1134 files belonging to 3 classes.


2026-08-26 00:01:58.322247: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-08-26 00:01:58.322280: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-08-26 00:01:58.322288: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-08-26 00:01:58.322321: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-26 00:01:58.322332: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [5]:
print(train_ds.class_names)

['freshapples', 'freshbanana', 'freshoranges']


In [6]:
print(f"TensorFlow Version: {tf.__version__}")
devices = tf.config.list_physical_devices()
print("Devices found:", devices)

gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print("Metal GPU is active and ready for your research!")
else:
    print("GPU not found. Check your tensorflow-metal installation.")

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

TensorFlow Version: 2.16.2
Devices found: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Metal GPU is active and ready for your research!
Num GPUs Available:  1


In [7]:
# Optimize input pipeline performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [8]:
# Data Augmentation Layer
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1)
])

### Model Structure

In [9]:
model = models.Sequential([
    layers.Input(shape=(96, 96, 1)),
    data_augmentation,
    layers.Rescaling(1./255),

    # Block 1: 16 filters
    layers.Conv2D(16, (3, 3), padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2, 2)),

    # Block 2: 32 filters
    layers.Conv2D(32, (3, 3), padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2, 2)),

    # Block 3: 64 filters for finer texture/edge extraction
    layers.Conv2D(64, (3, 3), padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2, 2)),

    # Global Average Pooling replaces Flatten to prevent overfitting
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(3, activation='softmax')
])

In [10]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 96, 96, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 96, 96, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 96, 96, 16)     │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 96, 96, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 96, 96, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 48, 48, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 48, 32)     │         4,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 48, 48, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 48, 48, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 24, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,811 (100.82 KB)

 Trainable params: 25,587 (99.95 KB)

 Non-trainable params: 224 (896.00 B)

In [12]:
# Callbacks
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_fruit_classifier.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

### Training

In [13]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[lr_scheduler, checkpoint]
    )

Epoch 1/15


/Users/haroon/miniconda3/envs/fruit_env/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-08-26 00:01:59.090183: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.7306 - loss: 0.6660
Epoch 1: val_accuracy improved from None to 0.55996, saving model to best_fruit_classifier.keras

Epoch 1: finished saving model to best_fruit_classifier.keras
149/149 ━━━━━━━━━━━━━━━━━━━━ 9s 51ms/step - accuracy: 0.7306 - loss: 0.6660 - val_accuracy: 0.5600 - val_loss: 1.0008 - learning_rate: 0.0010
Epoch 2/15
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8002 - loss: 0.5135
Epoch 2: val_accuracy improved from 0.55996 to 0.58730, saving model to best_fruit_classifier.keras

Epoch 2: finished saving model to best_fruit_classifier.keras
149/149 ━━━━━━━━━━━━━━━━━━━━ 6s 40ms/step - accuracy: 0.8002 - loss: 0.5135 - val_accuracy: 0.5873 - val_loss: 1.2669 - learning_rate: 0.0010
Epoch 3/15
148/149 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8174 - loss: 0.4813
Epoch 3: val_accuracy improved from 0.58730 to 0.82628, saving model to best_fruit_classifier.keras

Epoch 3: finished saving model to best_fru

In [16]:
best_model = tf.keras.models.load_model('best_fruit_classifier.keras')
print("\n--- Evaluation on Best Saved Model ---")
loss, accuracy = best_model.evaluate(val_ds)
print(f"Final Validation Accuracy: {accuracy * 100:.2f}%\n")


--- Evaluation on Best Saved Model ---
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8607 - loss: 0.3170
Final Validation Accuracy: 86.07%



In [ ]:
model.evaluate(val_ds)

36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8377 - loss: 0.4392


[0.4392027258872986, 0.8377425074577332]

## Quantization

In [17]:
print(help(tf.lite.TFLiteConverter))

Help on class TFLiteConverterV2 in module tensorflow.lite.python.lite:

class TFLiteConverterV2(TFLiteFrozenGraphConverterV2)
 |  TFLiteConverterV2(funcs, trackable_obj=None)
 |
 |  Converts a TensorFlow model into TensorFlow Lite model.
 |
 |  Attributes:
 |    optimizations: Experimental flag, subject to change. Set of optimizations to
 |      apply. e.g {tf.lite.Optimize.DEFAULT}. (default None, must be None or a
 |      set of values of type `tf.lite.Optimize`)
 |    representative_dataset: A generator function used for integer quantization
 |      where each generated sample has the same order, type and shape as the
 |      inputs to the model. Usually, this is a small subset of a few hundred
 |      samples randomly chosen, in no particular order, from the training or
 |      evaluation dataset. This is an optional attribute, but required for full
 |      integer quantization, i.e, if `tf.int8` is the only supported type in
 |      `target_spec.supported_types`. Refer to `tf.lite

### Quantization with Edge Impulse and TFLite

In [18]:
# The process was done through Edge Impulse Api and tflite.